# STAT 4845/5845 Lecture 6: areal data, median 2024 single family sale price in the Connecticut towns

Runtime > Change runtime type > R before running.

## Setup

Areal data come as polygons with attributes. The `sf` package reads, stores and plots them, and `CARBayes` fits the Bayesian model at the end. Install the system libraries sf needs, then both packages as prebuilt binaries from Posit Package Manager, once per session.

In [ ]:
system("apt-get install -y -qq libgdal-dev libgeos-dev libproj-dev libudunits2-dev", intern = TRUE)  # system libraries for sf

codename <- system("lsb_release -cs", intern = TRUE)  # Ubuntu release of this Colab machine
options(repos = c(P3M = paste0("https://packagemanager.posit.co/cran/__linux__/", codename, "/latest")))
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(), paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"])))  # needed, or the server sends source code instead of binaries

install.packages(c("sf", "CARBayes"), Ncpus = 2)  # binaries, including all dependencies of CARBayes
install.packages("CARBayes", repos = "https://cloud.r-project.org", type = "source")  # CARBayes itself again from source: its binary does not load on Colab

library(sf)

## Data

One file, shared on Google Drive: the 169 Connecticut towns as polygons, with the median 2024 single family sale price (dollars), the number of sales, the town centroid, and the ACS 2020-2024 median household income, population, area and density.

In [ ]:
download.file("https://drive.google.com/uc?export=download&id=1uAfgLYtNFgTvUz1vL4kEESPlhhUL6mnF", "ct_towns_2024.rds", mode = "wb")

ct <- readRDS("ct_towns_2024.rds")

## Explore

An `sf` object is a data frame with one row per region and a geometry column holding the polygon.

In [ ]:
class(ct)

In [ ]:
nrow(ct)  # number of towns

In [ ]:
names(ct)

In [ ]:
ct  # the first rows, the geometry column at the end

In [ ]:
st_crs(ct)$Name  # coordinate reference system, longitude and latitude

In [ ]:
st_geometry(ct)[[1]]  # the polygon of the first town, a list of boundary coordinates

In [ ]:
summary(ct$median_price)  # 18 towns without a listed 2024 sale

In [ ]:
ct$town[is.na(ct$median_price)]

## Map the towns

The polygons alone, then each attribute as a choropleth map.

In [ ]:
plot(st_geometry(ct), border = "grey40")
text(ct$lon, ct$lat, ct$town, cex = 0.3)

In [ ]:
pal <- function(k) hcl.colors(k, "Viridis")

plot(ct["median_price"], main = "Median sale price 2024, dollars", pal = pal, border = "grey40")  # white: no sales data

In [ ]:
plot(ct["n_sales"], main = "Number of single family sales 2024", pal = pal, border = "grey40")

In [ ]:
plot(ct["median_income"], main = "Median household income, dollars", pal = pal, border = "grey40")

In [ ]:
plot(ct["density"], main = "Population density, people per km2", pal = pal, border = "grey40")

## Neighbors

Two towns are neighbors when their polygons touch, sharing an edge or a corner (queen contiguity). `st_touches` works this out from the geometry and returns, for each town, the row numbers of its neighbors.

In [ ]:
n <- nrow(ct)

In [ ]:
nb <- st_touches(ct)  # queen neighbors: share an edge or corner

In [ ]:
nb[[which(ct$town == "Tolland")]]  # the neighbors of Tolland, by row number

In [ ]:
ct$town[nb[[which(ct$town == "Tolland")]]]

In [ ]:
i <- which(ct$town == "Tolland")
fill <- rep("white", n)
fill[nb[[i]]] <- "#EE6921"  # the neighbors in orange
fill[i] <- "#004369"  # Tolland in blue
plot(st_geometry(ct), col = fill, border = "grey40")
text(ct$lon[nb[[i]]], ct$lat[nb[[i]]], ct$town[nb[[i]]], cex = 0.7)
text(ct$lon[i], ct$lat[i], ct$town[i], cex = 0.7, col = "white")

## Proximity matrix W

Build W entry by entry: w_ij = 1 if towns i and j are neighbors, 0 otherwise.

In [ ]:
W <- matrix(0, n, n)
for (i in 1:n) W[i, nb[[i]]] <- 1  # w_ij = 1 if towns i and j share a border

In [ ]:
dim(W)

In [ ]:
isSymmetric(W)

In [ ]:
i <- which(ct$town == "Tolland")

In [ ]:
W[i, ]  # the row of Tolland, mostly zeros

In [ ]:
which(W[i, ] == 1)  # the same four neighbors as nb

In [ ]:
table(rowSums(W))  # number of neighbors per town

In [ ]:
ct$town[rowSums(W) == max(rowSums(W))]  # the town with the most neighbors

## Moran's I

Prices in thousand dollars. The 18 towns without a sale are left out: z holds the deviations from the mean for the 151 towns with data, and Wo is W restricted to those towns.

In [ ]:
y <- ct$median_price / 1000  # thousand dollars

In [ ]:
obs <- !is.na(y)  # towns with a listed 2024 sale

In [ ]:
z <- y[obs] - mean(y[obs])  # deviations from the mean

In [ ]:
Wo <- W[obs, obs]

In [ ]:
I <- (sum(obs) / sum(Wo)) * sum(z * (Wo %*% z)) / sum(z^2)  # Moran's I
I

In [ ]:
-1 / (sum(obs) - 1)  # expected value under no spatial autocorrelation

## Permutation test

Shuffle the prices among the towns, W fixed, and recompute I each time.

In [ ]:
set.seed(1)
I_perm <- replicate(999, {
  zp <- sample(z)
  (sum(obs) / sum(Wo)) * sum(zp * (Wo %*% zp)) / sum(zp^2)
})

hist(I_perm, breaks = 40, col = "#79AFBA", border = "white", xlim = range(c(I_perm, I)), main = "Moran's I of 999 permutations", xlab = "I")
abline(v = I, lwd = 3, col = "#BE2D2D")

In [ ]:
(sum(I_perm >= I) + 1) / (999 + 1)  # p value

## Local Moran's I and the LISA cluster map

I_i = n z_i (sum_j w_ij z_j) / sum_i z_i^2, one value per town. Its significance comes from a conditional permutation test: z_i stays fixed, its neighbors are drawn from the other towns.

In [ ]:
no <- sum(obs)

In [ ]:
s2 <- sum(z^2)

In [ ]:
I_local <- as.vector(no * z * (Wo %*% z) / s2)  # local Moran's I, one value per town

In [ ]:
head(data.frame(town = ct$town[obs], z = round(z, 1), I_local = round(I_local, 2)), 10)

In [ ]:
set.seed(1)
p_local <- sapply(1:no, function(i) {
  sims <- replicate(999, no * z[i] * sum(sample(z[-i], sum(Wo[i, ]))) / s2)  # neighbors drawn from the other towns
  (min(sum(sims >= I_local[i]), sum(sims <= I_local[i])) + 1) / (999 + 1)  # two sided
})

In [ ]:
sum(p_local < 0.05)  # towns with a significant local I

In [ ]:
lag <- as.vector(Wo %*% z) / rowSums(Wo)  # average deviation of the neighbors

In [ ]:
quad <- ifelse(z >= 0 & lag >= 0, "High-High", ifelse(z < 0 & lag < 0, "Low-Low", ifelse(z >= 0, "High-Low", "Low-High")))

In [ ]:
cl <- ifelse(p_local < 0.05, quad, "Not significant")

In [ ]:
table(cl)

In [ ]:
ct$town[obs][cl %in% c("High-Low", "Low-High")]  # spatial outliers

In [ ]:
lcols <- c("High-High" = "#BE2D2D", "Low-Low" = "#004369", "High-Low" = "#F2A9A9", "Low-High" = "#9CC3D5", "Not significant" = "#EDEDED")
fill <- rep("#7A7A7A", n)  # no data
fill[obs] <- lcols[cl]
plot(st_geometry(ct), col = fill, border = "grey45", main = "LISA cluster map, p < 0.05")
legend("bottomright", legend = c(names(lcols), "No data"), fill = c(lcols, "#7A7A7A"), border = "grey45", bty = "n", cex = 0.8)

## Moran scatterplot

Horizontal axis: z_i. Vertical axis: the average z_j over the neighbors of town i. The slope of the fitted line is Moran's I with row standardized weights.

In [ ]:
plot(z, lag, pch = 16, col = lcols[quad], xlab = "Deviation from the mean, z (thousand dollars)", ylab = "Average deviation of the neighbors")
abline(h = 0, v = 0, col = "grey60")
abline(lm(lag ~ z), lwd = 3, col = "#D55E00")
text(max(z), c(max(lag), min(lag)), c("High-High", "High-Low"), adj = 1, cex = 0.9)
text(min(z), c(min(lag) / 2, max(lag)), c("Low-Low", "Low-High"), adj = 0, cex = 0.9)

In [ ]:
coef(lm(lag ~ z))[2]  # the slope

In [ ]:
Wso <- Wo / rowSums(Wo)  # row standardized

In [ ]:
(sum(obs) / sum(Wso)) * sum(z * (Wso %*% z)) / sum(z^2)  # Moran's I with row standardized weights

## Model

$Y_i$ is the log median sale price in town $i$, for the $n = 151$ towns with data. The CAR model:

$$
\mathbf{Y} \sim N_n\{\boldsymbol{\mu}, \sigma^2 (\mathbf{D} - \rho \mathbf{W})^{-1}\},
\qquad
\mu_i = \beta_0 + \beta_1 \, \text{Income}_i + \beta_2 \, \text{Density}_i,
$$

where $\mathbf{W}$ is the proximity matrix of the 151 towns, $\mathbf{D}$ is the diagonal matrix of their numbers of neighbors $m_i$, Income is the median household income in thousand dollars and Density the population density in thousand people per km$^2$. Equivalently, for each town given the others,

$$
Y_i \mid \mathbf{Y}_{-i} \sim N\left(\mu_i + \frac{\rho}{m_i} \sum_{j=1}^{n} w_{ij} (Y_j - \mu_j), \; \frac{\sigma^2}{m_i}\right).
$$

Parameters: $\boldsymbol{\beta} = (\beta_0, \beta_1, \beta_2)^\top$, $\sigma^2 > 0$, $-1 < \rho < 1$.

## Model fitting: maximum likelihood

As in Lecture 5: the mean function $\boldsymbol{\mu}$ by least squares first, then $\boldsymbol{\theta} = (\sigma^2, \rho)^\top$ by maximum likelihood on the residuals $\mathbf{r} = \mathbf{Y} - \widehat{\boldsymbol{\mu}}$. The negative log likelihood

$$
\ell(\boldsymbol{\theta}) = \frac{n}{2} \log(2\pi) + \frac{1}{2} \log|\boldsymbol{\Sigma}| + \frac{1}{2} \mathbf{r}^\top \mathbf{Q} \mathbf{r},
\qquad
\mathbf{Q} = \boldsymbol{\Sigma}^{-1} = \frac{1}{\sigma^2} (\mathbf{D} - \rho \mathbf{W}),
$$

is written from $\mathbf{Q}$ directly, with no matrix inversion: with the Cholesky factorization $\mathbf{Q} = \mathbf{L} \mathbf{L}^\top$, $\log|\boldsymbol{\Sigma}| = -2 \sum_i \log L_{ii}$. `optim` minimizes $\ell(\boldsymbol{\theta})$ numerically.

In [ ]:
d <- ct[obs, ]  # the 151 towns with data

In [ ]:
n <- nrow(d)

In [ ]:
y <- log(d$median_price)  # log median sale price

In [ ]:
x1 <- d$median_income / 1000  # thousand dollars

In [ ]:
x2 <- d$density / 1000  # thousand people per km2

In [ ]:
Do <- diag(rowSums(Wo))  # number of neighbors of each town, Wo is W among the 151 towns

Mean function by least squares.

In [ ]:
fit_mean <- lm(y ~ x1 + x2)

summary(fit_mean)$coefficients

In [ ]:
r <- residuals(fit_mean)  # y - mu hat

In [ ]:
(n / sum(Wo)) * sum(r * (Wo %*% r)) / sum(r^2)  # Moran's I of the residuals

Maximum likelihood for $\sigma^2$ and $\rho$.

In [ ]:
nll <- function(theta) {  # negative log likelihood
  sigma2 <- exp(theta[1])  # variance, log scale keeps it positive
  rho <- tanh(theta[2])  # spatial dependence, tanh keeps it between -1 and 1
  cat(sigma2, rho, "\n"); flush.console()  # flush so Colab shows each line
  Q <- (Do - rho * Wo) / sigma2  # precision matrix, written from D and W, no inversion
  L <- t(chol(Q))  # Q = L L^T, L lower triangular
  logdet <- -2 * sum(log(diag(L)))  # log|Sigma| = -log|Q|
  nll_value <- 0.5 * n * log(2 * pi) + 0.5 * logdet +
    0.5 * sum(r * (Q %*% r))
  return(nll_value)
}

# start from a guess
fit_mle <- optim(c(log(0.1), atanh(0.5)), nll)

In [ ]:
c(exp(fit_mle$par[1]), tanh(fit_mle$par[2]))  # sigma^2 and rho

In [ ]:
-fit_mle$value  # log likelihood

Independent model for comparison: $\rho = 0$, so $\boldsymbol{\Sigma} = \sigma^2 \mathbf{D}^{-1}$.

In [ ]:
nll_ind <- function(theta) {  # negative log likelihood with rho = 0
  sigma2 <- exp(theta)
  Q <- Do / sigma2
  L <- t(chol(Q))
  logdet <- -2 * sum(log(diag(L)))
  0.5 * n * log(2 * pi) + 0.5 * logdet + 0.5 * sum(r * (Q %*% r))
}

fit_ind <- optim(log(0.1), nll_ind, method = "Brent", lower = log(1e-4), upper = log(10))

In [ ]:
exp(fit_ind$par)  # sigma^2

In [ ]:
c(independent = -fit_ind$value, CAR = -fit_mle$value)  # log likelihoods

## Bayesian approach

Route 2 for the Gaussian data: the hierarchical model with CAR random effects, as in the slides but with a Gaussian data model. For $i = 1, \ldots, n$,

$$
\begin{aligned}
Y_i \mid \mu_i &\sim N(\mu_i, \nu^2) && \text{Data model} \\
\mu_i &= \mathbf{x}_i^\top \boldsymbol{\beta} + u_i && \text{Linear predictor} \\
\mathbf{u} \mid \tau^2, \rho &\sim N_n\left[\mathbf{0}, \tau^2 \{\rho (\mathbf{D} - \mathbf{W}) + (1 - \rho) \mathbf{I}_n\}^{-1}\right] && \text{Process model: CAR} \\
\boldsymbol{\beta}, \tau^2, \nu^2, \rho &\sim \text{prior distributions} && \text{Parameter model}
\end{aligned}
$$

The spatial random effect $u_i$ carries the spatial dependence, and $\nu^2$ is the variance of the independent part. The process model is the Leroux form of the CAR prior used by the `CARBayes` package: $\rho = 1$ gives the intrinsic CAR (ICAR) of the slides, and $\rho = 0$ gives independent effects. Default priors: $\boldsymbol{\beta} \sim N(\mathbf{0}, 10^5 \mathbf{I})$, $\tau^2, \nu^2 \sim \text{Inverse-Gamma}(1, 0.01)$, $\rho \sim \text{Uniform}(0, 1)$.

## Model fitting: MCMC

`CARBayes` samples from the posterior distribution of $\boldsymbol{\beta}$, $\tau^2$, $\nu^2$, $\rho$ and $\mathbf{u}$ given $\mathbf{Y}$ by MCMC. The estimate of each unknown is its posterior mean, and a 95% credible interval gives its uncertainty. The first draws, the burn-in, are discarded; the rest are thinned to reduce autocorrelation.

In [ ]:
library(CARBayes)  # installed in the setup cell

In [ ]:
set.seed(1)
system.time(fit_bayes <- S.CARleroux(y ~ x1 + x2, family = "gaussian", W = Wo, burnin = 2000, n.sample = 12000, thin = 5, verbose = FALSE))

In [ ]:
fit_bayes$summary.results  # posterior means and 95% credible intervals

The MCMC draws are kept in `fit_bayes$samples`. Trace plots should look like noise around a stable level.

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4)  # wide figure for two panels
par(mfrow = c(1, 2))
plot(as.numeric(fit_bayes$samples$rho), type = "l", xlab = "Iteration", ylab = expression(rho))
plot(as.numeric(fit_bayes$samples$tau2), type = "l", xlab = "Iteration", ylab = expression(tau^2))
options(repr.plot.width = 7, repr.plot.height = 7)

In [ ]:
hist(as.numeric(fit_bayes$samples$rho), breaks = 40, col = "#79AFBA", border = "white", main = expression("Posterior of " * rho), xlab = expression(rho))

Regression coefficients: posterior means next to the least squares estimates.

In [ ]:
cbind(posterior_mean = colMeans(fit_bayes$samples$beta), least_squares = coef(fit_mean))

Posterior distributions of the regression coefficients: histograms of the MCMC draws, with the posterior mean in red and the 95% credible interval in blue.

In [ ]:
beta_post <- fit_bayes$samples$beta  # 2000 draws by 3 coefficients
colnames(beta_post) <- c("Intercept", "Income", "Density")

dim(beta_post)

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 4)  # wide figure for three panels
par(mfrow = c(1, 3))
for (j in 1:3) {
  hist(beta_post[, j], breaks = 30, col = "#79AFBA", border = "white", main = colnames(beta_post)[j], xlab = "")
  abline(v = mean(beta_post[, j]), lwd = 3, col = "#BE2D2D")  # posterior mean
  abline(v = quantile(beta_post[, j], c(0.025, 0.975)), lwd = 2, lty = 2, col = "#004369")  # 95% credible interval
}
options(repr.plot.width = 7, repr.plot.height = 7)

In [ ]:
mean(beta_post[, "Density"] > 0)  # posterior probability that the density effect is positive

The posterior means of $u_1, \ldots, u_n$ give the smoothed map of the spatial effects.

In [ ]:
d$u_hat <- colMeans(fit_bayes$samples$phi)  # posterior mean of u_i, one per town

In [ ]:
m <- max(abs(d$u_hat))
plot(d["u_hat"], main = "Posterior mean of the spatial effects, log price", pal = function(k) hcl.colors(k, "Blue-Red 3"), breaks = seq(-m, m, length.out = 11), border = "grey40")  # white at zero

## Prediction of the 18 towns without a sale

Write the CAR model for all 169 towns, $\mathbf{Y} \sim N_{169}\{\boldsymbol{\mu}, \sigma^2 (\mathbf{D} - \rho \mathbf{W})^{-1}\}$, with $\mathbf{W}$ and $\mathbf{D}$ of all 169 towns, and split the towns into the 151 observed ($o$) and the 18 missing ($m$). For a Gaussian vector with precision matrix $\mathbf{Q}$, the conditional distribution of the missing part given the observed part is

$$
\mathbf{Y}_m \mid \mathbf{Y}_o \sim N_{18}\left\{\boldsymbol{\mu}_m - \mathbf{Q}_{mm}^{-1} \mathbf{Q}_{mo} (\mathbf{Y}_o - \boldsymbol{\mu}_o), \; \mathbf{Q}_{mm}^{-1}\right\},
$$

where $\mathbf{Q}_{mm}$ is the $18 \times 18$ block of $\mathbf{Q}$ for the missing towns and $\mathbf{Q}_{mo}$ the $18 \times 151$ block between the missing and the observed towns. Only the small block $\mathbf{Q}_{mm}$ is solved, never the full $169 \times 169$ matrix. The conditional mean is the prediction and the diagonal of $\mathbf{Q}_{mm}^{-1}$ the prediction variance. The parameters are the maximum likelihood estimates from the 151 towns.

In [ ]:
y_all <- log(ct$median_price)  # all 169 towns, NA for the 18 without a sale

In [ ]:
X_all <- cbind(1, ct$median_income / 1000, ct$density / 1000)

In [ ]:
mu_all <- as.vector(X_all %*% coef(fit_mean))  # mean function at all 169 towns

In [ ]:
sigma2 <- exp(fit_mle$par[1])
rho <- tanh(fit_mle$par[2])

In [ ]:
D_all <- diag(rowSums(W))  # W is the proximity matrix of all 169 towns

In [ ]:
Q <- (D_all - rho * W) / sigma2  # precision matrix of all 169 towns

In [ ]:
Q_mm <- Q[!obs, !obs]  # 18 by 18
Q_mo <- Q[!obs, obs]  # 18 by 151

dim(Q_mm)

In [ ]:
pred <- mu_all[!obs] - as.vector(solve(Q_mm, Q_mo %*% (y_all[obs] - mu_all[obs])))  # conditional mean, log price

In [ ]:
pvar <- diag(solve(Q_mm))  # conditional variance

Back to dollars: $\exp$ of the predicted log price is the predicted median price, and $\exp$ of the log prediction $\pm 1.96$ standard deviations gives a 95% prediction interval.

In [ ]:
data.frame(town = ct$town[!obs], predicted = round(exp(pred) / 1000), lower = round(exp(pred - 1.96 * sqrt(pvar)) / 1000), upper = round(exp(pred + 1.96 * sqrt(pvar)) / 1000))  # thousand dollars

In [ ]:
ct$price_hat <- ct$median_price / 1000
ct$price_hat[!obs] <- exp(pred) / 1000

plot(ct["price_hat"], main = "Median sale price 2024, thousand dollars, 18 towns predicted", pal = pal, border = "grey40")

Bayesian prediction with `CARBayes`: pass all 169 towns with `NA` for the missing prices. The missing values are treated as unknowns and sampled along with the parameters, so each gets a posterior predictive distribution,

$$
p(\mathbf{Y}_m \mid \mathbf{Y}_o) = \int p(\mathbf{Y}_m \mid \mathbf{Y}_o, \boldsymbol{\beta}, \tau^2, \nu^2, \rho, \mathbf{u}) \, p(\boldsymbol{\beta}, \tau^2, \nu^2, \rho, \mathbf{u} \mid \mathbf{Y}_o) \, d(\boldsymbol{\beta}, \tau^2, \nu^2, \rho, \mathbf{u}),
$$

which averages the prediction over the posterior of the parameters, so the parameter uncertainty is included. The MCMC draws of $\mathbf{Y}_m$ are samples from it: their mean is the prediction and their 2.5% and 97.5% quantiles the 95% prediction interval.

In [ ]:
set.seed(1)
system.time(fit_all <- S.CARleroux(y_all ~ X_all[, 2] + X_all[, 3], family = "gaussian", W = W, burnin = 2000, n.sample = 12000, thin = 5, verbose = FALSE))

In [ ]:
Y_miss <- fit_all$samples$Y  # 2000 draws of the 18 missing log prices

dim(Y_miss)

In [ ]:
data.frame(town = ct$town[!obs], MLE = round(exp(pred) / 1000), Bayes = round(exp(as.vector(colMeans(Y_miss))) / 1000),
  lower = round(exp(as.vector(apply(Y_miss, 2, quantile, 0.025))) / 1000), upper = round(exp(as.vector(apply(Y_miss, 2, quantile, 0.975))) / 1000))  # thousand dollars